In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.cm as cm
import math as math

In [ ]:
def sample(pot, beta=1.0, delta_t = 0.001, N=10000, seed=42):
    rng = np.random.default_rng(seed=seed)
     
    X = [-0.6, 1.2]
    dim = 2 
    traj = []
    save = 100
    tlist = []
    for i in tqdm(range(N)):
        b = rng.normal(size=(dim,))
        X = X - pot.gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b
        if i % save==0:
            traj.append(X)
            tlist.append(i * delta_t)

    return np.array(tlist), np.array(traj)    

In [ ]:
class CurvedChannel: 
    def __init__(self, *argv):
        self.dim = 2
        self.x_domain = [-3, 3.0]
        self.y_domain = [-3, 3.0]
        self.x0 = [-1, 0]
        self.v_min_max = [-3,5]
        self.name = 'curved channel'
        self.contour_levels = [-3.0, -2.0, -1.0, 0, 1.5, 2.0, 3.0]

        self.min_A = [-1.0, 0]
        self.min_B = [1.0, 0]

        self.density_max = 0.1

        self.eps = 0.2

    def V(self, X):
        tmp1 = (X[0]**4 + X[1]**4) / 15.0 
        tmp2 = 4.0 * math.exp(-0.5 * (X[0]+2)**2 - 3.0 * (X[1])**2)
        tmp3 = 4.0 * math.exp(-0.5 * (X[0]-2)**2 - 3.0 * (X[1])**2)
        tmp4 = 2.0 * math.exp(-1.0 / self.eps * (X[0]**2/4 + X[1] * 0.5 - 1)**2) * (math.tanh(X[1]-0.2) + 1) * 0.5
        tmp5 = 6.0 * math.exp(-0.5 * X[0]**2 - 0.2 * (X[1]+1.0)**2)
        tmp6 = 0.5 * math.exp(-1.0 * X[0]**2 - 1.0 * (X[1]-2.0)**2)
        s = tmp1 - tmp2 - tmp3 - tmp4 + tmp5 + tmp6

        return s
        
    def gradV(self, X):

        tmp1 = (X[0]**4 + X[1]**4) / 15.0 
        tmp2 = math.exp(-0.5 * (X[0]+2)**2 - 3.0 * (X[1])**2)
        tmp3 = math.exp(-0.5 * (X[0]-2)**2 - 3.0 * (X[1])**2)
        tmp4 = math.exp(-1.0 / self.eps * (X[0]**2/4 + X[1] * 0.5 - 1)**2) 
        tmp5 = math.exp(-0.5 * X[0]**2 - 0.2 * (X[1]+1.0)**2)
        tmp6 = math.exp(-1.0 * X[0]**2 - 1.0 * (X[1]-2.0)**2)

        dVx = 4 * X[0]**3 / 15 + 4.0 * tmp2 * (X[0]+2) + 4.0 * tmp3 * (X[0]-2) \
              + 2.0 / self.eps * (X[0]**2/4 + X[1] * 0.5 - 1) * X[0] * tmp4 * (math.tanh(X[1]-0.2) + 1) * 0.5\
              - 6.0 * X[0] * tmp5 - 1.0 * X[0] * tmp6 
        
        dVy = 4 * X[1]**3 / 15 + 24 * tmp2 * X[1] + 24 * tmp3 * X[1] \
              - 1.0 * tmp4 * (1-math.tanh(X[1]-0.2)**2) \
              + 2.0 / self.eps * (X[0]**2/4 + X[1] * 0.5 - 1) * tmp4 * (math.tanh(X[1]-0.2) + 1) * 0.5 \
              - 2.4 * (X[1] + 1.0) * tmp5 - 1.0 * (X[1]-2.0) * tmp6

        return np.array((dVx, dVy))


In [ ]:
pot = CurvedChannel()

In [ ]:
beta = 1.8
N = 1000000

tlist, trajectory = sample(pot, beta=beta, delta_t=0.01, N=N)

print ('shape of the trajectory data:', trajectory.shape)

In [ ]:
fig = plt.figure(figsize=(6,6))

ax1 = fig.add_subplot(1, 1, 1)

nx = ny = 200

dx = (pot.x_domain[1] - pot.x_domain[0]) / nx
dy = (pot.y_domain[1] - pot.y_domain[0]) / ny
gridx = np.linspace(pot.x_domain[0], pot.x_domain[1], nx)
gridy = np.linspace(pot.y_domain[0], pot.y_domain[1], ny)
x_plot = np.outer(gridx, np.ones(ny)) 
y_plot = np.outer(gridy, np.ones(nx)).T 

# get grid points
x2d = np.concatenate((x_plot.reshape(nx * ny, 1), y_plot.reshape(nx * ny, 1)), axis=1)

# evaluate potential on grid points
pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)
# plot contour lines of the potential
contours = ax1.contour(x_plot, y_plot, pot_on_grid, levels=pot.contour_levels, cmap='coolwarm')

# plot the potential and its contour lines
im = ax1.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=pot.v_min_max[0], vmax=pot.v_min_max[1])
contours = ax1.contour(x_plot, y_plot, pot_on_grid,  pot.contour_levels)
ax1.clabel(contours, inline=True, fontsize=13,colors='black')

ax1.set_aspect('equal')
ax1.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax1.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax1.set_title("Meuller-Brown potential",fontsize=15)

# scatter plot of the trajectory data
ax1.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c='k', s=4)

ax1.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax1.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax1.set_title('trajectory')

plt.show()

In [ ]:
# Define the model
class Committor(nn.Module):
    def __init__(self):
        super(Committor, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(2, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        output = self.net(x)
        return output

In [ ]:
def data_in_A(x):
    ac = [-2, 0]
    idx = ((x-ac)**2).sum(axis=1) < 1
    return idx, x[idx,:]

def data_in_B(x):
    bc = [2, 0]
    idx = ((x-bc)**2).sum(axis=1) < 1
    return idx, x[idx,:]

In [ ]:
idx_A, X_A = data_in_A(trajectory)

idx_B, X_B = data_in_B(trajectory)

idx_other = np.logical_not(np.logical_or(idx_A, idx_B))

X_other = trajectory[idx_other,:]

plt.scatter(X_A[:,0], X_A[:,1], c='b')
plt.scatter(X_B[:,0], X_B[:,1], c='r')
plt.scatter(X_other[:,0], X_other[:,1], c='g')

print (X_A.shape, X_B.shape, X_other.shape)

plt.xlim([pot.x_domain[0], pot.x_domain[1]])
plt.ylim([pot.y_domain[0], pot.y_domain[1]])

X_A = torch.tensor(X_A, dtype=torch.float32)
X_B = torch.tensor(X_B, dtype=torch.float32)

In [ ]:
n_epochs = 100
batch_size = 128

train_losses = []

model = Committor()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(torch.tensor(X_other, dtype=torch.float32), batch_size=batch_size, shuffle=True)

for epoch in tqdm(range(n_epochs)):  # Max epochs
    model.train()
    epoch_train_loss = 0
    
    # Train on minibatches
    for batch_data in train_loader:
        optimizer.zero_grad()
        
        batch_data.requires_grad_()
        
        q = model(batch_data)
        
        g_grad = torch.autograd.grad(outputs=q.sum(), inputs=batch_data, retain_graph=True)[0]
        loss_1 = 1.0 / (beta * N) * (g_grad**2).sum() 
        
        loss_2 = (model(X_A)**2).mean()
        loss_3 = ((model(X_B)-1.0)**2).mean()
                
        # reconstruction loss
        loss = loss_1 + loss_2 + loss_3
        
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * batch_data.size(0)  # Accumulate loss
     
    # Compute average training loss for the epoch
    epoch_train_loss /= len(train_loader.dataset)
    
    # Store losses for plotting
    train_losses.append(epoch_train_loss)

# Plot training and validation loss curves
plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Over Epochs")
plt.legend()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10,6))

ax0 = fig.add_subplot(1, 2, 1)
ax1 = fig.add_subplot(1, 2, 2)

nx = ny = 200

dx = (pot.x_domain[1] - pot.x_domain[0]) / nx
dy = (pot.y_domain[1] - pot.y_domain[0]) / ny
gridx = np.linspace(pot.x_domain[0], pot.x_domain[1], nx)
gridy = np.linspace(pot.y_domain[0], pot.y_domain[1], ny)
x_plot = np.outer(gridx, np.ones(ny)) 
y_plot = np.outer(gridy, np.ones(nx)).T 

x2d = np.concatenate((x_plot.reshape(nx * ny, 1), y_plot.reshape(nx * ny, 1)), axis=1)

pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)

# plot the potential and its contour lines
im = ax0.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=pot.v_min_max[0], vmax=pot.v_min_max[1])
contours = ax0.contour(x_plot, y_plot, pot_on_grid,  pot.contour_levels)
ax0.clabel(contours, inline=True, fontsize=13,colors='black')

ax0.set_aspect('equal')
ax0.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax0.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax0.set_title("Meuller-Brown potential",fontsize=15)

with torch.no_grad():
    grid_tensor = torch.tensor(x2d, dtype=torch.float32)
    # evaluate encoder on grid points
    q_values = model(grid_tensor).numpy().reshape(nx, ny)

# show trajectory data    
ax1.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c='k', s=4)

ax1.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax1.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax1.set_aspect('equal')
ax1.set_title("Committor",fontsize=15)

im = ax1.pcolormesh(x_plot, y_plot, q_values, cmap='coolwarm', vmin=0, vmax=1)
contours = ax1.contour(x_plot, y_plot, q_values,  10)
ax1.clabel(contours, inline=True, fontsize=13,colors='black')

plt.show()